In [29]:
# imports 

import sys 
import ast
import numpy as np
import pandas as pd

import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer

# find text_functions.py in the src folder
sys.path.append('../src')
from text_functions import leniency_pos, leniency_neg, teaching_pos, teaching_neg
import os

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\noaht\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


##### Loading the cleaned data

In [30]:
# load the cleaned dataset from 01_cleaning
reviews_df = pd.read_csv('../data/interim/rmp_clean.csv')

# empty cleaned reviews are read as missing, so they are filled with empty strings
reviews_df['review_clean'] = reviews_df['review_clean'].fillna('')

# tags are a list saved as text, turn them back into a real list
reviews_df['tags'] = reviews_df['tags'].apply(ast.literal_eval)

print("Reviews:", len(reviews_df))

Reviews: 56597


##### Inspecting word frequency

In [31]:
# share of reviews containing each word or phrase in the four lists
word_lists = {'leniency +': leniency_pos, 'leniency -': leniency_neg,
              'teaching +': teaching_pos, 'teaching -': teaching_neg}

rows = []
for list_name, word_list in word_lists.items():
    for word in word_list:
        # \b marks word boundaries so 'easy' does not match 'easygoing'
        share = reviews_df['review_clean'].str.contains(r'\b' + word + r'\b', regex=True).mean()
        rows.append({'list': list_name, 'word': word, 'pct_of_reviews': round(share * 100, 2)})

word_counts_df = pd.DataFrame(rows).sort_values(['list', 'pct_of_reviews'], ascending=[True, False])

# most common words in each list
display(word_counts_df.groupby('list').head(5))

# words that never appear (candidates to remove or replace)
print(word_counts_df[word_counts_df['pct_of_reviews'] == 0])

,list,word,pct_of_reviews
0,leniency +,easy,16.06
9,leniency +,extra credit,2.25
2,leniency +,easy class,1.85
8,leniency +,curve,1.20
1,leniency +,easy grader,0.88
48,leniency -,not_easy,1.34
32,leniency -,tough grader,1.30
27,leniency -,harsh,0.83
33,leniency -,hard grader,0.65
34,leniency -,strict,0.62


           list               word  pct_of_reviews
20   leniency +        not_harshly             0.0
22   leniency +          not_picky             0.0
25   leniency +     not_subjective             0.0
26   leniency +        not_deducts             0.0
46   leniency -    grade deflation             0.0
50   leniency -       not_generous             0.0
52   leniency -          not_chill             0.0
84   teaching +        not_unclear             0.0
85   teaching +    not_disorganize             0.0
86   teaching +    not_unorganized             0.0
88   teaching +       not_monotone             0.0
89   teaching +         not_mumble             0.0
91   teaching +        not_tangent             0.0
92   teaching +     not_unprepared             0.0
93   teaching +      not_pointless             0.0
95   teaching +     not_irrelevant             0.0
118  teaching -       not_explains             0.0
128  teaching -       not_thorough             0.0
129  teaching -     not_insight

##### TF-IDF weights for the dictionary words

In [32]:
# every word and phrase in the four lists
all_terms = sorted(set(leniency_pos + leniency_neg + teaching_pos + teaching_neg))

# vocabulary is limited to the dictionary words, but idf still comes from all reviews so rare words weigh more
# ngram_range covers phrases up to 4 words, token_pattern keeps not_ words together as one token
tfidf = TfidfVectorizer(vocabulary=all_terms, ngram_range=(1, 4), token_pattern=r'(?u)\b[\w_]+\b',
                        sublinear_tf=True, norm=None)
tfidf_matrix = tfidf.fit_transform(reviews_df['review_clean'])

print("Matrix shape:", tfidf_matrix.shape)

Matrix shape: (56597, 133)


##### Positive, negative, and net scores

In [33]:
# adds up the tf-idf weights of a list's words for every review
def list_score(word_list):
    columns = [tfidf.vocabulary_[word] for word in word_list if word in tfidf.vocabulary_]
    return np.asarray(tfidf_matrix[:, columns].sum(axis=1)).ravel()

for concept, pos_list, neg_list in [('leniency', leniency_pos, leniency_neg), ('teaching', teaching_pos, teaching_neg)]:
    pos = list_score(pos_list)
    neg = list_score(neg_list)
    total = pos + neg

    # separate positive and negative weights (how much of each the review contains)
    reviews_df[f'{concept}_pos'] = pos
    reviews_df[f'{concept}_neg'] = neg

    # net weight, positive minus negative (bigger means more or stronger words in that direction)
    reviews_df[f'{concept}_net'] = pos - neg

    # net score from -1 (all negative words) to +1 (all positive words), 0 if no words matched
    reviews_df[f'{concept}_ratio'] = np.where(total > 0, (pos - neg) / np.where(total > 0, total, 1), 0)

    # 1 if the review used any word from either list, so "not mentioned" is separate from a neutral response
    reviews_df[f'{concept}_mentioned'] = (total > 0).astype(int)

# share of reviews that mention each concept, and the score distributions
print(reviews_df.loc[reviews_df['leniency_mentioned'] == 1, 'leniency_ratio'].describe().round(3))
print(reviews_df.loc[reviews_df['teaching_mentioned'] == 1, 'teaching_ratio'].describe().round(3))

print(reviews_df.loc[reviews_df['leniency_mentioned'] == 1, 'leniency_net'].describe().round(3))
print(reviews_df.loc[reviews_df['teaching_mentioned'] == 1, 'teaching_net'].describe().round(3))

count    14702.000
mean         0.509
std          0.838
min         -1.000
25%         -0.041
50%          1.000
75%          1.000
max          1.000
Name: leniency_ratio, dtype: float64
count    21636.000
mean         0.338
std          0.906
min         -1.000
25%         -1.000
50%          1.000
75%          1.000
max          1.000
Name: teaching_ratio, dtype: float64
count    14702.000
mean         2.122
std          6.148
min        -29.671
25%         -0.482
50%          2.829
75%          5.419
max         29.703
Name: leniency_net, dtype: float64
count    21636.000
mean         1.868
std          6.639
min        -27.375
25%         -4.251
50%          4.011
75%          5.303
max         46.299
Name: teaching_net, dtype: float64


##### Validating the scores

In [34]:
# only reviews that mention each concept
len_df = reviews_df[reviews_df['leniency_mentioned'] == 1]
teach_df = reviews_df[reviews_df['teaching_mentioned'] == 1]

# leniency should go down as the difficulty rating goes up (negative correlation)
print("Leniency vs difficulty:", round(len_df[['leniency_ratio', 'difficulty']].corr().iloc[0, 1], 3))

# leniency should be lower when the student tagged 'tough grader'
tough = len_df['tags'].apply(lambda tags: 'tough grader' in tags)
print("Leniency, tough grader tag:", round(len_df.loc[tough, 'leniency_ratio'].mean(), 3),
      "  without:", round(len_df.loc[~tough, 'leniency_ratio'].mean(), 3))

# teaching should be higher when the student tagged 'amazing lectures'
amazing = teach_df['tags'].apply(lambda tags: 'amazing lectures' in tags)
print("Teaching, amazing lectures tag:", round(teach_df.loc[amazing, 'teaching_ratio'].mean(), 3),
      "  without:", round(teach_df.loc[~amazing, 'teaching_ratio'].mean(), 3))



Leniency vs difficulty: -0.48
Leniency, tough grader tag: -0.429   without: 0.617
Teaching, amazing lectures tag: 0.911   without: 0.289


##### Save the NLP scores

In [35]:
# save the scored dataset
os.makedirs('../data/processed', exist_ok=True)
reviews_df.to_csv('../data/processed/rmp_scored.csv', index=False)
print("Saved", reviews_df.shape)

Saved (56597, 36)
